In [ ]:
import pdfplumber
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Path to your PDF
pdf_path = '../rsc/docs/sample.pdf'

# Open the PDF
with pdfplumber.open(pdf_path) as pdf:
    # Iterate through each page
    for page_num, page in enumerate(pdf.pages, start=1):
        print(f"\n--- Page {page_num} ---")
        
        # Extract tables from the page
        tables = page.extract_tables()

        if tables:
            print(f"Found {len(tables)} tables on Page {page_num}")
            for i, table in enumerate(tables):
                print(f"\nTable {i+1}:")
                for row in table:
                    print(row)  # Print each row of the table

            # Debug: Visualize the table regions
            print(f"Visualizing table regions on Page {page_num}")
            image = page.to_image()  # Get a page image
            fig, ax = plt.subplots(figsize=(10, 12))

            # Show the page image
            ax.imshow(image.original)

            # Draw rectangles around the tables
            for table in page.find_tables():
                bbox = table.bbox  # Bounding box of the table
                rect = Rectangle(
                    (bbox[0], bbox[1]),  # Bottom-left corner
                    bbox[2] - bbox[0],  # Width
                    bbox[3] - bbox[1],  # Height
                    linewidth=2,
                    edgecolor='red',
                    facecolor='none'
                )
                ax.add_patch(rect)

            ax.set_title(f"Page {page_num} - Table Regions")
            plt.show()
        else:
            print(f"No tables found on Page {page_num}")

In [30]:
metrics = {
        "Tempo Effettivo": None,
        "Possesso CUS Torino": None,
        "Territorio CUS Torino": None,
        "Punti d'Incontro": None,
        "Portatori (Dominanti + Avanzanti + Non Avanzanti)": None,
        "Tackle Breaks": None,
        "Passaggi (Positivi + Negativi)": None,
        "Offload (Positivi + Negativi + Difensori Battuti)": None,
        "Kicks in Play (Positivo + Negativo)": None,
    }

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        
        # Extract "Tempo Effettivo"
        if "TEMPO EFFETTIVO" in text:
            tempo_start = text.find("TEMPO EFFETTIVO") + len("TEMPO EFFETTIVO")
            metrics['Tempo Effettivo'] = text[tempo_start:].split()[0]
        # Extract "Possesso CUS Torino"
        if "POSSESSO" in text:
            possesso_lines = [line for line in text.split("\n") if "POSSESSO" in line]
            for line in possesso_lines:
                if "CUS TORINO" in line:
                    metrics["Possesso CUS Torino"] = line.split()[-1].replace("%", "")
        
        # Extract "Territorio CUS Torino"
        if "TERRITORIO" in text:
            territorio_lines = [line for line in text.split("\n") if "TERRITORIO" in line]
            for line in territorio_lines:
                if "CUS TORINO" in line:
                    metrics["Territorio CUS Torino"] = line.split()[-1].replace("%", "")
        
        # Extract "Punti d'Incontro"
        if "P.I" in text:
            pi_lines = [line for line in text.split("\n") if "P.I" in line]
            metrics["Punti d'Incontro"] = sum(map(int, [s for line in pi_lines for s in line.split() if s.isdigit()]))
        
        # Extract Portatori and Tackle Breaks
        if "PORTATORI" in text:
            portatori_start = text.find("PORTATORI") + len("PORTATORI")
            portatori_data = text[portatori_start:].split("\n")
            metrics["Portatori (Dominanti + Avanzanti + Non Avanzanti)"] = sum(
                int(s) for s in portatori_data[0].split() if s.isdigit()
            )
            if "Tackle breaks" in text:
                metrics["Tackle Breaks"] = int(text.split("Tackle breaks")[1].split()[0])
        
        # Extract Passaggi
        if "PASSAGGI" in text:
            passaggi_start = text.find("PASSAGGI") + len("PASSAGGI")
            passaggi_data = text[passaggi_start:].split("\n")
            metrics["Passaggi (Positivi + Negativi)"] = sum(
                int(s) for s in passaggi_data[0].split() if s.isdigit()
            )
        
        # Extract Offload and Difensori Battuti
        if "OFFLOAD" in text:
            offload_start = text.find("OFFLOAD") + len("OFFLOAD")
            offload_data = text[offload_start:].split("\n")
            metrics["Offload (Positivi + Negativi + Difensori Battuti)"] = sum(
                int(s) for s in offload_data[0].split() if s.isdigit()
            )
        
        # Extract Kicks in Play
        if "Kicks in play" in text:
            kicks_start = text.find("Kicks in play") + len("Kicks in play")
            kicks_data = text[kicks_start:].split("\n")
            metrics["Kicks in Play (Positivo + Negativo)"] = sum(
                int(s) for s in kicks_data[0].split() if s.isdigit()
            )

In [31]:
metrics

{'Tempo Effettivo': '36:30',
 'Possesso CUS Torino': 'POSSESSO',
 'Territorio CUS Torino': None,
 "Punti d'Incontro": 0,
 'Portatori (Dominanti + Avanzanti + Non Avanzanti)': 141,
 'Tackle Breaks': 14,
 'Passaggi (Positivi + Negativi)': 0,
 'Offload (Positivi + Negativi + Difensori Battuti)': 6,
 'Kicks in Play (Positivo + Negativo)': 18}

In [38]:
print(pdf.pages[0].extract_table()[3])

[None, None, '', None, None, None, 'PRIMO TEMPO', None, None, 'SECONDO TEMPO', None, None, None, 'TOTALE', None, None, None, 'PRIMO TEMPO', 'SECONDO TEMPO', None, None, None, 'TOTALE']


In [24]:
with pdfplumber.open(pdf_path) as pdf:
    # Iterate through each page
    for page_num, page in enumerate(pdf.pages, start=1):
        print(f"\n--- Page {page_num} ---")
        
        # Extract table from the page
        table = page.extract_tables()
        print(table)
        break


--- Page 1 ---
[[['PETRARCA Serie A1 2024-2025 CUS TORINO\n30 28\nG9 Rugby Petrarca - CUS Torino 19-01-2025', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None], ['PETRARCA\nFioriti\nGoat\nMonti\nTelandro\nBonfiglio\nTognon\nGoldin\nLucas\nBisaglia\nOrmson\nFilippi\nTrez\nIaonnucci\nDella Silvestra\nBenvenuti\nBaldo\nZin\nPidone\nSala\nZapparoli\nRaccanello\nNardo\nGiaccarello', 'CUS TORINO\nValleise\nBau\nAraujo\nAndreica\nMastrodomenico\nPerrone\nRotger\nQuaglia\nLa Terza\nZanatta\nMomicchioli\nBolognesi\nTorres\nCivita\nReeves E.\nCataldi\nChecchini\nMuciaccia\nRiccardi\nFerrari\nTruffa\nAmbrosi\nTelloni', 'TEMPO EFFETTIVO', None, None, None, None, None, None, None, None, None, '36:30', None, None, None, 'INGRESSI NEI 22', None, None, None, None, None, None], [None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, 'PETRARCA\nPRIMO TEMPO SECONDO TEMPO TOTALE', No

In [26]:
import pandas as pd
import seaborn as sns

# Parse into a DataFrame
df = pd.DataFrame(table)
df.dropna(inplace=True)

In [ ]:


# Extract specific metrics
match_details = df.iloc[0, 0]
score = match_details.split('\n')[1]
teams = match_details.split('\n')[0].split('Serie A1')[0].strip()

# Clean metrics
metrics = df.iloc[1:]
metrics.columns = ['Metric', 'Value']

# Visualize possession
possession = metrics[metrics['Metric'].str.contains('Possession')]
team_data = possession['Value'].str.split(' ', expand=True)
team_data.columns = ['Team', 'Pct']
team_data['Pct'] = team_data['Pct'].str.replace('%', '').astype(float)

# Plot possession distribution
sns.barplot(data=team_data, x='Team', y='Pct')
plt.title("Possession Distribution")
plt.show()

# Display cleaned metrics
print(metrics)

In [39]:
def debug_lines(text):
    """Debugging function to print lines for inspection."""
    print("\nExtracted Lines:")
    for i, line in enumerate(text.split("\n")):
        print(f"{i + 1}: {line}")

def extract_metrics(pdf_path):
    metrics = {
        "Tempo Effettivo": None,
        "Possesso CUS Torino": None,
        "Territorio CUS Torino": None,
        "Punti d'Incontro": None,
        "Portatori (Dominanti + Avanzanti + Non Avanzanti)": None,
        "Tackle Breaks": None,
        "Passaggi (Positivi + Negativi)": None,
        "Offload (Positivi + Negativi + Difensori Battuti)": None,
        "Kicks in Play (Positivo + Negativo)": None,
    }

    # Open the PDF with pdfplumber
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            text = page.extract_text()

            # Debug: Print lines from the page
            print(f"\n--- Page {page_num} ---")
            debug_lines(text)
    return metrics

In [43]:
with pdfplumber.open(pdf_path) as pdf:
    page_1 = pdf.pages[0]
    text = page_1.extract_text()
    debug_lines(text)
    tables = page_14.extract_tables()

    for table in tables:
        # Convert the table to a DataFrame for easier processing
        df = pd.DataFrame(table)
        print(df)


Extracted Lines:
1: PETRARCA Serie A1 2024-2025 CUS TORINO
2: 30 28
3: G9 Rugby Petrarca - CUS Torino 19-01-2025
4: PETRARCA CUS TORINO INGRESSI NEI 22
5: TEMPO EFFETTIVO 36:30
6: PETRARCA
7: Fioriti Valleise
8: PRIMO TEMPO SECONDO TEMPO TOTALE PRIMO TEMPO SECONDO TEMPO TOTALE
9: Goat Bau 1° QUARTO 2° QUARTO 3° QUARTO 4° QUARTO
10: BALL IN PLAY 0:33 2:12 2:45
11: 10:01 7:33 10:45 8:10
12: Monti Araujo
13: BALL IN PLAY 36:30 n.ingressi 4 6 10
14: TOT 17:34 TOT 18:56
15: Telandro Andreica 1°TEMPO 2°TEMPO
16: METE 2 2 4
17: LAVORO / RIPOSO 0,74 0,74 0,74
18: Bonfiglio Mastrodomenico
19: 3 PUNTI 0 0 0
20: N.SEQUENZE 26 27 53
21: Tognon Perrone
22: Durata media ingressi 0:16.55
23: LUNGHEZZA SEQUENZE
24: Goldin Rotger Punti per ingresso 2
25: Lucas Quaglia
26: CUS TORINO
27: < 10 sec 10 - 30 sec 31 - 60 sec 61 - 90 sec 91 - 120 sec > 120 sec
28: Bisaglia La Terza PRIMO TEMPO SECONDO TEMPO TOTALE
29: 11 12 22 3 1 4
30: Ormson Zanatta BALL IN PLAY 2:39 4:40 7:19
31: Filippi Momicchioli n. in